# Runs brainvisa preprocessing

This notebook creates the whole brainvisa preprocessing pipeline to feed to deep learning algorithms.
Note that you need brainvisa installed or you need to run the server under the brainvisa singularity.

You also need to have access to the HCP database (the HCP folder lies as subfolder of root_unsupervised) and a path to the root of the supervised folder (/neurospin for persons present at neurospin)

# Sets root directories

Contrary to the other notebooks, this one relies on data that are outside the deep_folding/data folder.

In [ ]:
# This is the root of the HCP directory
# Could be either /tgcc, or /nfs/tgcc, for example
root_unsupervised = '/tgcc'  

# This is the root of the supervised directory
# Could be either /neuropsin, or /nfs/neurospin, for example
root_supervised = '/neurospin' 

# Imports

General imports

In [ ]:
import sys
import os
from os.path import join
import glob
import json
import inspect

Deep_folding imports

In [ ]:
from deep_folding.brainvisa import generate_skeletons
from deep_folding.brainvisa import generate_ICBM2009c_transforms
from deep_folding.brainvisa import resample_files
from deep_folding.brainvisa import compute_bounding_box
from deep_folding.brainvisa import compute_mask
from deep_folding.brainvisa import generate_crops
print(inspect.getfile(compute_bounding_box))
print(inspect.getfile(generate_crops))

Constants

In [ ]:
_ALL_SUBJECTS = -1

In [ ]:
out_voxel_size = 1

In [ ]:
number_subjects_supervised = 10 # Number of subjects for which we determine the box. We can set it to _ALL_SUBJECTS

In [ ]:
number_subjects = 10 # Number of subjects for which we generate the crops. We can set it to _ALL_SUBJECTS

# Creates useful functions

In [ ]:
def check_directory(directory_path):
    """Checks directory path and returns absolute path"""
    directory_path = os.path.abspath(directory_path)
    if os.path.isdir(directory_path):
        print((directory_path + ' is a directory'))
    else:
        print((directory_path + ' does not exist or is not a directory.'))
    return directory_path

# Variables used by all sub-computations

The following boolean variables decide which pprocessing to run:

In [ ]:
run_bbox = True  # If set to True, it generates new bounding boxes
run_mask = True  # If set to True, it generates new masks
run_crop = True  # If set to True, it generates crops

We now assign path names and other user-specific variables.

The unsupervised source directory is where the unsupervised database lies. It contains the morphologist analysis subfolder ANALYSIS/3T_morphologist


In [ ]:
unsupervised_src_dir = check_directory(join(root_unsupervised, 'hcp', 'ANALYSIS/3T_morphologist'))

The supervised source directories are where lies the database that has been manually labelled. It is a list of full pathes towards the manually labelled datasets.

In [ ]:
human_supervised_dir = join(root_supervised, 'dico/data/bv_databases/human')
supervised_src_dir = [check_directory(join(human_supervised_dir, 'pclean/all'))
                     ]
path_to_graph = ["t1mri/t1/default_analysis/folds/3.3/base2018_manual"
                 ]

# Generates bounding boxes

### User variables

In [ ]:
bbox_dir = check_directory(join(root_supervised, 'dico/data/deep_folding/test', 'bbox'))

In [ ]:
mask_dir = check_directory(join(root_supervised, 'dico/data/deep_folding/current/mask/2mm'))

Lists the sulci of the left side that we want to analyze:

In [ ]:
sulci_left = ['S.T.s.ter.asc.ant.', 'S.T.s.ter.asc.post.']

Lists the sulci of the right side that we want to analyze:

In [ ]:
sulci_right = ['S.T.s.ter.asc.ant.', 'S.T.s.ter.asc.post.']

### Generates bounding boxes (actual program)

We first call crop_definition help as if called from a command line:

In [ ]:
args = "--help"
argv = args.split(' ')
compute_bounding_box.main(argv)

In [ ]:
print(sulci_left, sulci_right)

We now run the actial program.
This saves as json files in bbox_dir the bounding box characteristics:

In [ ]:
if run_bbox:
    for sulcus in sulci_left:
        compute_bounding_box.compute_bounding_box(
            src_dir=supervised_src_dir, 
            path_to_graph=path_to_graph,
            bbox_dir=bbox_dir,
            sulcus=sulcus,
            side='L',
            number_subjects=number_subjects_supervised,
            out_voxel_size=out_voxel_size)
    for sulcus in sulci_right:
        compute_bounding_box.compute_bounding_box(
            src_dir=supervised_src_dir, 
            path_to_graph=path_to_graph,
            bbox_dir=bbox_dir,
            sulcus=sulcus,
            side='R',
            number_subjects=number_subjects_supervised,
            out_voxel_size=out_voxel_size)

# Generates crops

### User variables 

In [ ]:
interp = 'nearest'

In [ ]:
crop_dir = check_directory(join(root_supervised, 'dico/data/deep_folding/test', 'crops'))

### Generates crops (actual program)

We now save in {crop_dir}/Lcrops and {crop_dir}/Rcrops the actual crops based on bounding boxes:

In [ ]:
skeleton_raw_dir = check_directory(join(root_supervised, 'dico/data/deep_folding/test', 'skeletons/raw'))

In [ ]:
skeleton_1mm_dir = check_directory(join(root_supervised, 'dico/data/deep_folding/test', 'skeletons/1mm'))

In [ ]:
transform_dir = check_directory(join(root_supervised, 'dico/data/deep_folding/test', 'transform'))

In [ ]:
if run_crop:
    generate_skeletons.generate_skeletons(
        src_dir=unsupervised_src_dir,
        skeleton_dir=skeleton_raw_dir,
        side='L',
        number_subjects=number_subjects)
    generate_skeletons.generate_skeletons(
        src_dir=unsupervised_src_dir,
        skeleton_dir=skeleton_raw_dir,
        side='R',
        number_subjects=number_subjects)

In [ ]:
if run_crop:
    generate_ICBM2009c_transforms.generate_ICBM2009c_transforms(
        src_dir=unsupervised_src_dir,
        transform_dir=transform_dir,
        side='L',
        number_subjects=number_subjects)
    generate_ICBM2009c_transforms.generate_ICBM2009c_transforms(
        src_dir=unsupervised_src_dir,
        transform_dir=transform_dir,
        side='R',
        number_subjects=number_subjects)

In [ ]:
if run_crop:
    resample_files.resample_files(
        src_dir=skeleton_raw_dir,
        input_type='skeleton',
        resampled_dir=skeleton_1mm_dir,
        transform_dir=transform_dir,
        side='L',
        number_subjects=number_subjects)
    resample_files.resample_files(
        src_dir=skeleton_raw_dir,
        input_type='skeleton',
        resampled_dir=skeleton_1mm_dir,
        transform_dir=transform_dir,
        side='R',
        number_subjects=number_subjects)
    print("Done")

In [ ]:
if run_crop:
    # Runs on left hemisphere
    generate_crops.generate_crops(
        src_dir=skeleton_1mm_dir,
        crop_dir=crop_dir,
        bbox_dir=bbox_dir,
        cropping_type='bbox',
        list_sulci=sulci_left,
        side='L',
        number_subjects=number_subjects)
    # Runs on right hemisphere
    generate_crops.generate_crops(
        src_dir=skeleton_1mm_dir,
        crop_dir=crop_dir,
        bbox_dir=bbox_dir,
        cropping_type='bbox',
        list_sulci=sulci_right,
        side='R',
        number_subjects=number_subjects)
    print("Done")

We now sort generated files (we do it in date order; indeed, we may find older files in folder as we don't suppress files before writing new ones):

In [ ]:
files = glob.glob(f"{crop_dir}/Lcrops/*.nii.gz")
files.sort(key=os.path.getmtime, reverse=True)
print("\n".join(files[:number_subjects]))

In [ ]:
files = glob.glob(f"{crop_dir}/Rcrops/*.nii.gz")
files.sort(key=os.path.getmtime, reverse=True)
print("\n".join(files[:number_subjects]))

### Generates mask-based crops

In [ ]:
if run_mask:
    for sulcus in sulci_left:
        compute_mask.compute_mask(
            src_dir=supervised_src_dir, 
            path_to_graph=path_to_graph,
            mask_dir=mask_dir,
            sulcus=sulcus,
            side='L',
            number_subjects=number_subjects_supervised,
            out_voxel_size=out_voxel_size)
    for sulcus in sulci_right:
        compute_mask.compute_mask(
            src_dir=supervised_src_dir, 
            path_to_graph=path_to_graph,
            mask_dir=mask_dir,
            sulcus=sulcus,
            side='R',
            number_subjects=number_subjects_supervised,
            out_voxel_size=out_voxel_size)
    print("Done")

In [ ]:
if run_crop:
    # Runs on left hemisphere
    generate_crops.generate_crops(
        src_dir=skeleton_1mm_dir,
        crop_dir=crop_dir,
        mask_dir=mask_dir,
        list_sulci=sulci_left,
        side='L',
        cropping_type='mask',
        number_subjects=number_subjects)
    # Runs on right hemisphere
    generate_crops.generate_crops(
        src_dir=skeleton_1mm_dir,
        crop_dir=crop_dir,
        mask_dir=mask_dir,
        list_sulci=sulci_left,
        side='R',
        cropping_type='mask',
        number_subjects=number_subjects)
    print("Done")